# 12 — Create the Trace Observability Genie Agent

**Jackson and Jackson HR Digital**

Creates (or updates) a Genie Agent over the trace analytics views from notebook `11`, so anyone can
ask plain-English questions about how the Hire Right agent behaves — no MLflow UI or SQL required.

### What this notebook does
1. Loads the agent definition from `12_trace_genie_space.json` (instructions, column descriptions,
   synonyms, joins, example SQL, measures, and benchmarks)
2. Creates the Genie Agent, or updates it in place if it already exists
3. Asks a test question through the Genie Conversation API

### Example questions
- Which tool is the slowest on average, and have any tools failed?
- How many agent requests did we handle each week, and what was the p95 latency?
- How often did the agent have to ask the user for missing candidate scores?
- What is the pass rate for each LLM judge?

### Prerequisites
- `11_trace_analytics_views.ipynb` has run (views exist)
- A SQL warehouse ID (`warehouse_id` widget)

## Setup

In [ ]:
%pip install databricks-sdk python-dotenv -q

In [ ]:
dbutils.widgets.text("trace_catalog", "bldemos",                      "UC Catalog (traces)")
dbutils.widgets.text("trace_schema",  "hrd_2030_agent_observability", "UC Schema (traces)")
dbutils.widgets.text("warehouse_id",  "",                             "SQL Warehouse ID")

In [ ]:
import json, os, time, requests

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_root  = "/Workspace" + notebook_path.rsplit("/notebooks", 1)[0]
try:
    from dotenv import load_dotenv
    load_dotenv(f"{project_root}/.env")
except ImportError:
    pass

# Widget takes priority (job passes it); fall back to .env for interactive runs
trace_catalog = dbutils.widgets.get("trace_catalog") or os.getenv("TRACE_CATALOG", "bldemos")
trace_schema  = dbutils.widgets.get("trace_schema")  or os.getenv("TRACE_SCHEMA", "hrd_2030_agent_observability")
warehouse_id  = dbutils.widgets.get("warehouse_id")  or os.getenv("DATABRICKS_WAREHOUSE_ID", "")
if not warehouse_id:
    raise ValueError("warehouse_id is required for the Genie Agent.")

ctx     = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host    = ctx.apiUrl().get().rstrip("/")
headers = {"Authorization": f"Bearer {ctx.apiToken().get()}", "Content-Type": "application/json"}

print(f"Views        : {trace_catalog}.{trace_schema}.v_agent_*")
print(f"Warehouse ID : {warehouse_id}")

## Load Definition & Create / Update the Genie Agent

In [ ]:
genie_json_path = f"/Workspace{os.path.dirname(notebook_path)}/12_trace_genie_space.json"
with open(genie_json_path) as f:
    genie_meta = json.load(f)

serialized_space = (
    json.dumps(genie_meta["serialized_space"])
    .replace("__CATALOG__", trace_catalog)
    .replace("__SCHEMA__", trace_schema)
)
title = genie_meta["title"]

payload = {
    "warehouse_id":     warehouse_id,
    "title":            title,
    "description":      genie_meta["description"],
    "serialized_space": serialized_space,
}

# Reuse an existing agent with the same title so re-runs update in place
genie_space_id = ""
_resp = requests.get(f"{host}/api/2.0/genie/spaces", headers=headers, timeout=30)
_resp.raise_for_status()
for _s in _resp.json().get("spaces", []):
    if _s.get("title") == title:
        genie_space_id = _s["space_id"]
        break

if genie_space_id:
    resp = requests.patch(f"{host}/api/2.0/genie/spaces/{genie_space_id}", headers=headers, json=payload, timeout=60)
    action = "Updated"
else:
    resp = requests.post(f"{host}/api/2.0/genie/spaces", headers=headers, json=payload, timeout=60)
    action = "Created"
if resp.status_code not in (200, 201):
    raise RuntimeError(f"Genie API error {resp.status_code}: {resp.text[:1000]}")
genie_space_id = genie_space_id or resp.json()["space_id"]

print(f"✓ {action} Genie Agent: {title}")
print(f"  Space ID : {genie_space_id}")
print(f"  URL      : {host}/genie/rooms/{genie_space_id}")

## Test the Genie Agent

In [ ]:
test_question = "Which tool is the slowest on average, and have any tools failed?"
print(f"Q: {test_question}\n")

start = requests.post(
    f"{host}/api/2.0/genie/spaces/{genie_space_id}/start-conversation",
    headers=headers, json={"content": test_question}, timeout=60,
)
start.raise_for_status()
conv_id = start.json()["conversation_id"]
msg_id  = start.json()["message_id"]
poll_url = f"{host}/api/2.0/genie/spaces/{genie_space_id}/conversations/{conv_id}/messages/{msg_id}"

msg = {}
for _ in range(60):
    msg = requests.get(poll_url, headers=headers, timeout=30).json()
    if msg.get("status") in ("COMPLETED", "FAILED", "CANCELLED"):
        break
    time.sleep(3)

print(f"Status: {msg.get('status')}")
for att in msg.get("attachments", []):
    if att.get("text"):
        print(att["text"].get("content", ""))
    if att.get("query"):
        print("\nSQL:\n" + att["query"].get("query", ""))
        _res = requests.get(
            f"{poll_url}/attachments/{att['attachment_id']}/query-result", headers=headers, timeout=30
        ).json()
        _sr = _res.get("statement_response", {})
        _cols = [c["name"] for c in _sr.get("manifest", {}).get("schema", {}).get("columns", [])]
        for _row in _sr.get("result", {}).get("data_array", [])[:10]:
            print(dict(zip(_cols, _row)))

## Summary

The **Hire Right Agent — Trace Observability Genie** is live over the four `v_agent_*` views.
Because the views sit directly on the UC trace tables, every new production trace is queryable
in Genie immediately.

In [ ]:
try:
    dbutils.jobs.taskValues.set(key="trace_genie_space_id", value=genie_space_id)
except Exception:
    pass
dbutils.notebook.exit(genie_space_id)